In [5]:
import pandas as pd
import re
import json


input_path = "jty3_.txt"  

with open(input_path, "r", encoding="utf-8") as f:
    text = f.read()

pattern = r"(\/[^\n]+\/(normal|abnormal)\/[^\n]+)\s*[\s\S]*?(?:```json\s*([\s\S]*?)\s*```|(\{[\s\S]*?\}))"
matches = re.findall(pattern, text, re.DOTALL)

rows = []

for match in matches:
    
    path, truth, json_block, json_raw = match
    json_str = json_block if json_block else json_raw
    
    try:
        data = json.loads(json_str)

        is_scam = bool(data.get("is_scam"))

        if (is_scam and truth == "abnormal") or (not is_scam and truth == "normal"):
            result = True
        else:
            result = False

        rows.append({
            "filename": path.split("/")[-1].strip(),
            "truth": truth,
            "is_scam": data.get("is_scam"),
            "result": result,
            "is_correct": data.get("is_scam"),
            "confidence": data.get("confidence"),
            "risk": data.get("risk"),
            "evidence": "; ".join(data.get("evidence", [])),
            "explanation": data.get("explanation"),            
        })
    except json.JSONDecodeError as e:
        print(f"JSON 파싱 오류 발생 ({path}): {e}")

df = pd.DataFrame(rows, columns=["filename", "truth", "is_scam", "result", "confidence", "risk", "evidence", "explanation"])

output_path = "scam_results.csv"
df.to_csv(output_path, index=False, encoding="utf-8-sig")

print(f"CSV 파일 생성 완료: {output_path}")
print(df)


JSON 파싱 오류 발생 (/home/ubuntu/cybercop/video_20251104/abnormal/0034.mp4): Invalid control character at: line 5 column 81 (char 141)
CSV 파일 생성 완료: scam_results.csv
    filename     truth  is_scam  result  confidence  risk  \
0   0025.mp4    normal     True   False        0.95  high   
1   0011.mp4    normal    False    True        0.80   low   
2   0024.mp4    normal    False    True        0.65   low   
3   0004.mp4    normal     True   False        0.95  high   
4   0001.mp4    normal    False    True        0.65   low   
5   0006.mp4    normal     True   False        0.95  high   
6   0016.mp4    normal    False    True        0.45   low   
7   0005.mp4    normal    False    True        0.65   low   
8   0018.mp4    normal    False    True        0.85   low   
9   0023.mp4    normal    False    True        0.52   low   
10  0007.mp4    normal    False    True        0.95   low   
11  0014.mp4    normal    False    True        0.95   low   
12  0021.mp4    normal    False    True       

In [7]:
df.groupby(["truth", "result"]).agg({"filename":"count"})

filename
truth    result          
abnormal False         10
         True          16
normal   False          5
         True          23

In [8]:
df[df['result']==True]

,filename,truth,is_scam,result,confidence,risk,evidence,explanation
1,0011.mp4,normal,False,True,0.80,low,"알바하게 되면 하는 것; 이어폰 뺀 후; 아니요, 코로노바에 손을 대지 말고; 알바...",The video appears to be instructional content ...
2,0024.mp4,normal,False,True,0.65,low,유가리 없이 시리얼 먹는 법; 그러고; ereniket...?,The video describes how to eat cereal without ...
4,0001.mp4,normal,False,True,0.65,low,"MZ는 왜 이제アルバ리를 안 할까요?; 2025년 최저임금 인상 확정 10,030원...",The video appears to be a discussion about the...
6,0016.mp4,normal,False,True,0.45,low,추구 골タ임 먹이라고 고정받 퀴브를 맞춰버림; 유리저리 큐브를 돌려보며 계산을 마친...,视频似乎展示了拼图的过程，没有明显的内容要求观众投资、加入、支付或提供敏感数据。对话中使用的...
7,0005.mp4,normal,False,True,0.65,low,실머리 없는 알바생 투ughughugh; 야 야uk?; 야 야이야,The video does not show any clear manipulative...
8,0018.mp4,normal,False,True,0.85,low,가성비 한국; 그림 한부분만 보여주면; 이거를,The video focuses on showcasing affordable and...
9,0023.mp4,normal,False,True,0.52,low,"Guaranteed profit, extreme ROI, short-term ric...",The video primarily showcases cooking techniqu...
10,0007.mp4,normal,False,True,0.95,low,100만원 벌기; 7일 동안 단기알바 만으로 100만원 벌기; 기대만usic 뷰어와...,The video details a person's plan to earn mone...
11,0014.mp4,normal,False,True,0.95,low,날이 더워지면 친음 보시는 분들이 오셔서 음료를 다라고 오시거든요 😅😂; 어제도 음...,The video shows a person preparing and serving...
12,0021.mp4,normal,False,True,0.65,low,"이상한 돈을 받을 수 있대요; 한 달에 160만 원을 벌 수 있다; 1,160만원;...",The video appears to be informative and promot...


In [9]:
df[df['result']==False]

,filename,truth,is_scam,result,confidence,risk,evidence,explanation
0,0025.mp4,normal,True,False,0.95,high,우버 택시 받tees; 오빠 택시 50% 허트 프로모션 코드 설치 중...; 지금 ...,The video advertises a 'Uber Taxi' program wit...
3,0004.mp4,normal,True,False,0.95,high,보건증 필수; 손劲이 돼서 꼭 찍어줘; 사 nä… 왜 이거…,The video promotes performing a specific actio...
5,0006.mp4,normal,True,False,0.95,high,이러면서 그Nam 말朋友하게 시켜야겠다 했는데; 오늘 오늘 신청한 거 됐다고; 이제...,The video contains multiple indicators of scam...
22,0026.mp4,normal,True,False,0.85,high,우버 택시 받약; 우나 택시 50% 헌타 프로모션 코드 설치 중...; 우버 양 옆...,The video advertises a promotion for 'Uber Tax...
27,0017.mp4,normal,True,False,0.92,high,의대생이 시험지를 반대로집어두는 이유; 나를 수학 1등급으로 만들어주마,The video appears to provide tips on how stude...
32,0004.mp4,abnormal,False,False,0.82,low,The video showcases a variety of sandals arran...,The focus is entirely on presenting different ...
37,0005.mp4,abnormal,False,False,0.72,low,가상화폐 구매대행 모집 한 달이면 마음에 드는 차를 살 수 있습니다.; 十字路口; ...,The video appears to be an informative or prom...
38,0018.mp4,abnormal,False,False,0.57,low,다들 박해; 음前所 콤비한; 俯瞰해,The chat conversation shows participants discu...
39,0007.mp4,abnormal,False,False,0.75,low,사람들을에게 공과 허망을 불어 넣어주는 사이비 교수; 사람들에게 공과 허망을 불어 ...,The video depicts a person in white demonstrat...
41,0031.mp4,abnormal,False,False,0.38,low,1:1 분의; 手游新規,"The text mentions '1:1 분의' and '手游新規', which s..."
